# PyTorch Micro Assignment

**Estimated time:** 45–60 minutes  
**Topics:** tensors, losses, autograd, linear layers, learning rates, Momentum, shapes and devices

This assignment contains six independent exercises. You may complete them in any order.

## Rules

- Use PyTorch for every numerical calculation.
- For “from scratch” functions, use basic tensor operations rather than the equivalent `nn` loss or layer.
- Use PyTorch library implementations only to verify your result.
- Write one or two sentences for every interpretation question.
- A GPU is not required.

There is no dataset and no complete model-training pipeline in this assignment.

## 0. Colab setup

Run the installation helper and imports first. The helper installs only packages that are missing.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "matplotlib": "matplotlib",
}

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )
    print("Installed:", ", ".join(missing_packages))
else:
    print("Required packages are already installed.")

In [ ]:
%matplotlib inline

import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.set_printoptions(precision=5, sci_mode=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

# Exercise 1: Implement MSE and BCE

## 1A. Mean squared error

Implement:

$$
\operatorname{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat y_i)^2.
$$

Requirements:

- Use tensor operations.
- Do not use a Python loop.
- Do not call `nn.MSELoss` inside your function.

In [ ]:
y_true_regression = torch.tensor([3.0, -0.5, 2.0, 7.0])
y_pred_regression = torch.tensor([2.5, 0.0, 2.0, 8.0])


def mse_from_scratch(y_true, y_pred):
    # TODO: Return the mean squared error using tensor operations.
    pass


custom_mse = mse_from_scratch(y_true_regression, y_pred_regression)
print("Custom MSE:", custom_mse)

In [ ]:
# Verification: this cell should run without an assertion error.
library_mse = nn.MSELoss()(y_pred_regression, y_true_regression)

assert torch.allclose(custom_mse, library_mse)
print("Library MSE:", library_mse)
print("MSE implementation verified.")

## 1B. Binary cross-entropy

Implement:

$$
\operatorname{BCE}
=-\frac{1}{n}\sum_{i=1}^{n}
\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].
$$

Requirements:

- Use tensor operations without a Python loop.
- Clamp probabilities so that logarithms never receive exactly 0 or 1.
- Do not call `nn.BCELoss` inside your function.

In [ ]:
y_true_binary = torch.tensor([1.0, 0.0, 1.0, 1.0])
y_pred_probability = torch.tensor([0.90, 0.20, 0.60, 0.70])


def binary_cross_entropy_from_scratch(y_true, probability, epsilon=1e-7):
    # TODO: Clamp probability to a numerically safe range.


    # TODO: Calculate and return the mean binary cross-entropy.
    pass


custom_bce = binary_cross_entropy_from_scratch(
    y_true_binary,
    y_pred_probability,
)
print("Custom BCE:", custom_bce)

In [ ]:
# Verification: this cell should run without an assertion error.
library_bce = nn.BCELoss()(y_pred_probability, y_true_binary)

assert torch.allclose(custom_bce, library_bce, atol=1e-6)
print("Library BCE:", library_bce)
print("BCE implementation verified.")

### Interpretation

Why must probabilities be protected from reaching exactly 0 or 1 before applying logarithms?

**Your answer:**

# Exercise 2: Verify autograd manually

Use:

$$
\widehat y=wx+b,\qquad L=(\widehat y-y)^2
$$

with:

$$
w=2,\quad b=1,\quad x=3,\quad y=10.
$$

Tasks:

1. Calculate the gradients manually using tensor expressions.
2. Calculate them using `loss.backward()`.
3. Verify that both methods agree.
4. Show that calling backward on another loss without clearing gradients causes accumulation.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)
target = torch.tensor(10.0)

# TODO: Calculate prediction and loss.


# TODO: Write the manual formulas for dL/dw and dL/db.


# TODO: Use backward() to calculate PyTorch's gradients.


print("Manual dL/dw:", manual_dw)
print("Autograd dL/dw:", w.grad)
print("Manual dL/db:", manual_db)
print("Autograd dL/db:", b.grad)

assert torch.allclose(w.grad, manual_dw)
assert torch.allclose(b.grad, manual_db)

In [ ]:
first_dw = w.grad.clone()

# TODO: Recalculate the same prediction and loss, then call backward again
# without clearing w.grad or b.grad.


print("Gradient after first backward:", first_dw)
print("Gradient after second backward:", w.grad)

### Interpretation

Why does a normal training loop call `optimizer.zero_grad()` before `loss.backward()`?

**Your answer:**

# Exercise 3: Recreate `nn.Linear`

For a linear layer:

$$
Z=XW+b.
$$

Tasks:

1. Calculate the output manually.
2. Create `nn.Linear(in_features=3, out_features=2)`.
3. Copy the supplied values into the layer.
4. Verify that the manual and library outputs match.

Remember that the mathematical $W$ below has shape `(in_features, out_features)`, while `nn.Linear.weight` has shape `(out_features, in_features)`.

In [ ]:
X = torch.tensor([
    [1.0, 2.0, -1.0],
    [0.5, -1.0, 3.0],
])

W = torch.tensor([
    [0.2, -0.4],
    [0.7, 0.3],
    [-0.5, 0.8],
])

bias = torch.tensor([0.1, -0.2])

# TODO: Calculate manual_output = X @ W + bias.


# TODO: Create a Linear layer with three inputs and two outputs.


with torch.no_grad():
    # TODO: Copy W and bias into the layer.
    # Hint: inspect the shape of linear.weight before copying.
    pass

library_output = linear(X)

print("Manual output:\n", manual_output)
print("nn.Linear output:\n", library_output)

assert torch.allclose(manual_output, library_output)

In [ ]:
# TODO: Calculate the parameter count manually.


library_parameter_count = sum(
    parameter.numel() for parameter in linear.parameters()
)

print("Manual parameter count:", manual_parameter_count)
print("PyTorch parameter count:", library_parameter_count)

assert manual_parameter_count == library_parameter_count

### Interpretation

Why must the supplied mathematical weight matrix be transposed before copying it into `nn.Linear.weight`?

**Your answer:**

# Exercise 4: Learning-rate experiment

Minimize:

$$
L(w)=(w-3)^2
$$

starting from $w=0$. Run 15 updates for:

$$
\eta\in\{0.01,\ 0.1,\ 0.9,\ 1.1\}.
$$

Use autograd to calculate the gradient, but update $w$ manually inside `torch.no_grad()`.

In [ ]:
def optimize_scalar(learning_rate, steps=15):
    # TODO: Create w=0 with gradient tracking.


    values = [w.item()]
    losses = []

    for _ in range(steps):
        # TODO: Calculate L(w)=(w-3)^2 and run backward.


        # TODO: Update w without adding the update operation to the graph.


        # TODO: Clear w.grad.


        values.append(w.item())
        losses.append(loss.item())

    final_loss = (w.detach() - 3) ** 2
    return values, losses, w.detach(), final_loss

In [ ]:
learning_rates = [0.01, 0.1, 0.9, 1.1]
learning_rate_results = {}

for learning_rate in learning_rates:
    result = optimize_scalar(learning_rate)
    learning_rate_results[learning_rate] = result
    _, _, final_w, final_loss = result
    print(
        f"lr={learning_rate:>4}: "
        f"final w={final_w.item():>10.4f}, "
        f"final loss={final_loss.item():>12.6f}"
    )

In [ ]:
# Plotting code is provided; interpret the trajectories.
plt.figure(figsize=(8, 4.5))

for learning_rate, (values, _, _, _) in learning_rate_results.items():
    plt.plot(values, marker="o", label=f"learning rate = {learning_rate}")

plt.axhline(3, color="black", linestyle="--", label="minimum: w=3")
plt.xlabel("Update")
plt.ylabel("w")
plt.title("Effect of learning rate on parameter updates")
plt.legend()
plt.show()

### Interpretation

Classify each learning rate as slow, fast, oscillating while converging, or diverging. Explain using the plotted sequence of $w$ values.

**Your answer:**

# Exercise 5: Two SGD and Momentum updates

Use the fixed gradient sequence:

$$
g_1=-6,qquad g_2=-4.8
$$

with:

$$
w_0=0,qquad \eta=0.1,qquad \beta=0.9,qquad v_0=0.
$$

Implement two updates for:

$$
\text{SGD: }w_t=w_{t-1}-\eta g_t
$$

and:

$$
\text{Momentum: }v_t=\beta v_{t-1}+g_t,qquad
w_t=w_{t-1}-\eta v_t.
$$

In [ ]:
gradients = torch.tensor([-6.0, -4.8])
initial_w = torch.tensor(0.0)
learning_rate = 0.1
beta = 0.9


def two_sgd_updates(initial_w, gradients, learning_rate):
    # TODO: Apply every gradient and return all parameter values,
    # including the initial value.
    pass


def two_momentum_updates(initial_w, gradients, learning_rate, beta):
    # TODO: Start velocity at zero.


    # TODO: Apply every Momentum update and return all parameter values
    # and all velocity values.
    pass


sgd_values = two_sgd_updates(initial_w, gradients, learning_rate)
momentum_values, velocities = two_momentum_updates(
    initial_w,
    gradients,
    learning_rate,
    beta,
)

print("SGD parameter values:", sgd_values)
print("Momentum parameter values:", momentum_values)
print("Momentum velocities:", velocities)

### Interpretation

Why is the second Momentum update larger than the second plain-SGD update when both gradients point in the same direction?

**Your answer:**

# Exercise 6: Tensor shape and device challenge

Start with:

```python
images = torch.rand(32, 1, 28, 28)
```

Complete every transformation and satisfy all assertions.

In [ ]:
images = torch.rand(32, 1, 28, 28)

# TODO: Flatten each image to produce shape (32, 784).


# TODO: Restore the flattened tensor to shape (32, 1, 28, 28).


# TODO: Remove only the channel dimension to produce (32, 28, 28).


# TODO: Add the channel dimension back to produce (32, 1, 28, 28).


# TODO: Move the restored images to `device`.


# TODO: Create four class labels using the dtype required by
# nn.CrossEntropyLoss.


assert flattened.shape == (32, 784)
assert restored.shape == (32, 1, 28, 28)
assert without_channel.shape == (32, 28, 28)
assert with_channel.shape == (32, 1, 28, 28)
assert images_on_device.device == device
assert labels.dtype == torch.long

print("All shape, device, and dtype checks passed.")

In [ ]:
predictions = torch.tensor([2, 0, 1, 2], device=device)

# TODO: Convert predictions to a NumPy array safely.


print("NumPy predictions:", predictions_numpy)
print("Type:", type(predictions_numpy))

### Interpretation

Why must a GPU tensor be moved to the CPU before converting it to NumPy?

**Your answer:**

# Submission checklist

- [ ] MSE matches `nn.MSELoss`.
- [ ] BCE matches `nn.BCELoss`.
- [ ] Manual and autograd gradients match.
- [ ] Gradient accumulation is demonstrated.
- [ ] Manual linear output matches `nn.Linear`.
- [ ] Manual and PyTorch parameter counts match.
- [ ] Four learning-rate behaviors are explained.
- [ ] SGD and Momentum updates are implemented.
- [ ] All tensor shape and dtype assertions pass.
- [ ] Every interpretation question has a one- or two-sentence answer.

The assignment does not require a report, dataset, GPU, or complete neural-network training pipeline.